In [ ]:
#1.导入库
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['font.sans-serif']=['SimHei']


In [ ]:
#2.导入数据
df=pd.read_csv('E:\GitHub\date analys-study\项目.房地产市值评估与预测\housing.csv')


In [ ]:
#3.数据概览
print('总记录数:',len(df))
print('字段数量:',len(df.columns))
df.info


In [ ]:
#4.数据清洗
df.drop(columns='url',inplace=True)#原地删除无用项
#处理缺失值
df=df.replace('暂无',np.nan)
df.dropna(inplace=True)
#检查重复值
df.duplicated().sum()
df.drop_duplicates(inplace=True)


In [ ]:
#数据类型转换
df['area']=df['area'].str.replace('㎡','').astype(float)
#售价的数据类型转换
df['price']=df['price'].str.replace('万','').astype(float)
#朝向的数据类型转换
df['toward']=df['toward'].astype('category')

In [ ]:
#单价的数据类型转换
from numpy import int64


df['unit']=df['unit'].str.replace('元/㎡','').astype(float)
df['year']=df['year'].str.replace('年建',"").astype(int64)
df.head()


In [ ]:
#异常值的处理
#房屋面积的异常处理
from regex import T


df=df[(df['area']>20)&(df['area']<600)]
#房屋售价的异常值处理
Q1=df['price'].quantile(0.25)
Q3=df['price'].quantile(0.75)
TQR=Q3-Q1
low_price=Q1-1.5*TQR
high_price=Q3+1.5*TQR
df[df['price']<low_price]
df[df['price']>high_price]
df=df[(df['price']>low_price)&(df['price']<high_price)]



In [ ]:
#5.新数据特征构造
#地区district
from re import I

from numpy import astype
from sklearn.metrics import root_mean_squared_error


df['district']=df['address'].str.split('·').str[0]
#楼层类型floor_type
df['floor_type']=df['floor'].str.split('(').str[0].astype('category')
def fun1(str1):
    if pd.isna(str1):
        return'未知'
    elif '低' in str1:
        return '低楼层'
    elif "中" in str1:
        return'中楼层'
    elif '高' in str1:
        return'高楼层'
    else:
        return'未知'
df['floor_type2']= df['floor'].apply(fun1).astype('category')
#是否是直辖市
# def fun2(str2):
#     if str2 in['北京','上海','天津','重庆']:
#         return True
#     else:
#         return False
df['是否是直辖市']=df['city'].apply(lambda x:1 if x in ['北京','上海','天津','重庆'] else 0)
df.sample(100)
#卧室的数量bedrooms
df['bedrooms']=df['rooms'].str.split('室').str[0].astype(int)
#客厅的数量livingrooms
#df["livingrooms"]=df['rooms'].str.split('室').str[1].str.split('厅').str[0].astype(int)
df["livingrooms"]=df['rooms'].str.extract(r'(\d+)厅').astype(int)
df.head()
#楼龄building_age
df['building_age']=2026-df['year']
df.sample(10)
#价格的分段price_labelspd
df['price_labels']=pd.cut(df['price'],bins=4,labels=['低价','中价','高价','豪华'])
df.sample(10)

In [ ]:
#6.问题分析及可视化
'''
哪些变量最影响房价,面积,楼层,房间哪个影响更大
分析主题:特征相关性
分析目标:了解房屋各特征值的线性影响
分组字段:无
指标/方法:皮尔逊相关系数
'''
#选择数值型特征
a=df[['price','area','unit','building_age']].corr()#相关系数
#对房价影响最大的几个因素排序
a['price'].sort_values(ascending=False)[1:]
#相关性的热力图
plt.figure(figsize=(5,5))
plt.title('房屋特征值相关性热力图')
sns.heatmap(a,cmap='coolwarm')#颜色设置

In [ ]:
'''
房价分布分布如何,是否存在极端值

'''
df.describe()
#房价分布直方图
plt.subplot(111)
plt.hist(df['price'],bins=10)

In [ ]:
sns.histplot(data=df,x='price',bins=10,kde=True)

In [ ]:
'''
朝向对价格影响
'''

df.groupby('toward').agg({
    'price':['mean','median'],
    'unit':'median',
    'building_age':'mean'
})

In [ ]:
#数据可视化
plt.figure(figsize=(14,5))
sns.boxplot(x='toward',y='price',data=df)